# 70 — M6 multi-step certificate

M6 専用ノートブックです。現時点では M5 の実環境受理を検証する fail-closed ゲートだけを収録します。M6 の全受理条件が満たされるまで `CERTIFIED` を出力しません。

In [ ]:
# 必ず最初に pytest を用意する。
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


In [ ]:
from pathlib import Path
import hashlib, json, os

PREVIOUS, CURRENT, RUN_ENV = 'M5', 'M6', 'VALIDATED_RG_M5_RUN_ID'
run_id = os.environ.get(RUN_ENV)
if not run_id:
    raise RuntimeError(f'{RUN_ENV} を受理済み M5 run ID に設定してください。M6 はまだ開始されません。')
if Path(run_id).name != run_id or not run_id.startswith('M5-'):
    raise RuntimeError('M5 run ID の形式が不正です。')
persist_root = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
acceptance_path = persist_root / 'runs' / run_id / 'reports' / 'M5_acceptance.json'
if not acceptance_path.is_file():
    raise RuntimeError(f'M5 acceptance artifact がありません: {acceptance_path}')
acceptance = json.loads(acceptance_path.read_text(encoding='utf-8'))
required = {'milestone': 'M5', 'phase': 'M5_COMPLETE', 'status': 'PASS'}
if any(acceptance.get(k) != v for k, v in required.items()):
    raise RuntimeError('M5 acceptance artifact の identity/status が不正です。')
# M5 may complete as ONE_STEP_CERTIFIED or as a verified NOT_CERTIFIED failure (q_cert≥1).
allowed_cert = {'NOT_CERTIFIED', 'ONE_STEP_CERTIFIED'}
if acceptance.get('certification_status') not in allowed_cert:
    raise RuntimeError(
        f"M5 certification_status must be one of {sorted(allowed_cert)}; "
        f"got {acceptance.get('certification_status')!r}."
    )
if not acceptance.get('accepted_for_next_milestone') == 'M6':
    raise RuntimeError('M5 acceptance is not marked accepted_for_next_milestone=M6.')
STAGE_GATE = {
    'current_milestone': CURRENT, 'predecessor': PREVIOUS, 'status': 'PREDECESSOR_ACCEPTED',
    'acceptance_path': str(acceptance_path),
    'acceptance_sha256': hashlib.sha256(acceptance_path.read_bytes()).hexdigest(),
    'implementation_status': 'LOCKED_PENDING_M6_IMPLEMENTATION_REVIEW',
    'm5_certification_status': acceptance.get('certification_status'),
    'certification_status': 'NOT_CERTIFIED',
}
STAGE_GATE


## M6 実装境界

直前セルが通っても M6 完了でも certification でもありません。multi-step enclosure、influence/tail/residual の合成、全 P-gate、証拠 hash chain、独立再検証、最終 acceptance report は未実装です。M5 の人間レビュー後に、このノートブックだけを拡張します。